# Scene point cloud by instance ID

Select a generated scene with its configuration and scene ID, then color its TLS or geometry-sampled point cloud by `instance_index`.

In [ ]:
from pathlib import Path
import colorsys
import json
import numpy as np
import matplotlib.pyplot as plt

# Parameters
scene_type = "alley"  # alley, dense_park, or sparse_park
scene_id = 1
point_cloud_variant = "tls_scan"  # tls_scan or geometry_sampled
tls_scan_id = 1

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "PythonBinding").is_dir():
    repo_root = repo_root.parent
if not (repo_root / "PythonBinding").is_dir():
    raise FileNotFoundError("Could not locate the EvoEngine repository root")

scene_base_names = {
    "alley": "AlleyTreeScene",
    "dense_park": "DenseParkScene",
    "sparse_park": "SparseParkScene",
}
if scene_type not in scene_base_names:
    raise ValueError(f"Unknown scene_type: {scene_type}")
scene_base_name = scene_base_names[scene_type]
scene_name = f"{scene_base_name}_{scene_id:03d}"
output_dir = repo_root / "out" / scene_base_name / scene_name
id_map_path = output_dir / f"{scene_name}_object_ids.json"
point_cloud_suffix = f"tls_scan_{tls_scan_id:02d}" if point_cloud_variant == "tls_scan" else point_cloud_variant
ply_path = output_dir / f"{scene_name}_{point_cloud_suffix}.ply"
if not id_map_path.is_file() or not ply_path.is_file():
    raise FileNotFoundError(f"Scene output is incomplete: {output_dir}")
ply_path

In [ ]:
def load_evoengine_scanner_ply(path):
    data = Path(path).read_bytes()
    marker = b"end_header\n"
    header_end = data.index(marker) + len(marker)
    header = data[:header_end].decode("ascii")
    point_count = int(next(line.split()[2] for line in header.splitlines() if line.startswith("element vertex ")))
    payload = memoryview(data)[header_end:]
    expected_size = point_count * 20
    if len(payload) != expected_size:
        raise ValueError(f"Expected {expected_size} payload bytes, found {len(payload)}")
    points = np.frombuffer(payload, dtype="<f4", count=point_count * 3).reshape(point_count, 3)
    type_index = np.frombuffer(payload, dtype="<i4", count=point_count, offset=point_count * 12)
    instance_index = np.frombuffer(payload, dtype="<i4", count=point_count, offset=point_count * 16)
    return points, type_index, instance_index

all_points, all_type_index, all_instance_index = load_evoengine_scanner_ply(ply_path)
id_map = json.loads(id_map_path.read_text(encoding="utf-8"))
print(f"Scene: {scene_name}")
print(f"Point cloud: {ply_path.name} ({len(all_points):,} points)")
for object_id, count in zip(*np.unique(all_instance_index, return_counts=True)):
    name = id_map.get(str(int(object_id)), {}).get("name", "unknown")
    print(f"  {int(object_id):4d}: {int(count):9,d}  {name}")

points = all_points
type_index = all_type_index
instance_index = all_instance_index

In [ ]:
def instance_color(object_id):
    if object_id == 1000:
        return (0.53, 0.56, 0.57)
    if object_id >= 1001:
        return (0.86, 0.49, 0.22)
    hue = (object_id * 0.61803398875) % 1.0
    return colorsys.hsv_to_rgb(hue, 0.72, 0.92)

rng = np.random.default_rng(1200)
max_points_per_instance = 30_000

def plot_instances(ax, mask, title, legend=False, bounds=None):
    for object_id in np.unique(instance_index[mask]):
        indices = np.flatnonzero(mask & (instance_index == object_id))
        if len(indices) > max_points_per_instance:
            indices = rng.choice(indices, max_points_per_instance, replace=False)
        shown = points[indices]
        name = id_map.get(str(int(object_id)), {}).get("name", "unknown")
        ax.scatter(shown[:, 0], shown[:, 2], shown[:, 1], s=0.7,
                   color=instance_color(int(object_id)), alpha=0.8,
                   label=f"{int(object_id)}: {name}")
    ax.set_title(title)
    ax.set_xlabel("X")
    ax.set_ylabel("Z")
    ax.set_zlabel("Y")
    selected = points[mask]
    if bounds is None and len(selected):
        bounds = (selected.min(axis=0), selected.max(axis=0))
    if bounds is not None:
        minimum, maximum = bounds
        ranges = np.maximum(maximum - minimum, 0.1)
        ax.set_xlim(minimum[0], maximum[0])
        ax.set_ylim(minimum[2], maximum[2])
        ax.set_zlim(minimum[1], maximum[1])
        ax.set_box_aspect((ranges[0], ranges[2], ranges[1]))
    if legend and np.any(mask):
        ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), markerscale=5, fontsize=8)

fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection="3d")
plot_instances(ax, np.ones(len(points), dtype=bool), f"{scene_name} - {len(points):,} points", legend=True)
plt.tight_layout()
plt.show()

In [ ]:
def ids_matching(predicate):
    return [int(object_id) for object_id, metadata in id_map.items() if predicate(metadata)]

big_tree_ids = ids_matching(lambda item: item.get("level") == "big_tree")
small_tree_ids = ids_matching(lambda item: item.get("level") == "small_tree")
bush_ids = ids_matching(lambda item: item.get("level") == "bush" or item.get("kind") == "bush")
artificial_ids = ids_matching(lambda item: item.get("kind") == "artificial")
artificial_ids.append(1000)

shared_bounds = (points.min(axis=0), points.max(axis=0))
fig = plt.figure(figsize=(18, 14))
for index, (object_ids, title) in enumerate([
    (big_tree_ids, "Big tree level"),
    (small_tree_ids, "Small tree level"),
    (bush_ids, "Bush level"),
    (artificial_ids, "Ground / building level"),
], 1):
    mask = np.isin(instance_index, object_ids)
    plot_instances(fig.add_subplot(2, 2, index, projection="3d"), mask, title, bounds=shared_bounds)
plt.tight_layout()
plt.show()

In [ ]:
# Open the full point cloud in an interactive Open3D desktop viewer.
import open3d as o3d

viewer_colors = np.empty((len(points), 3), dtype=np.float64)
for object_id in np.unique(instance_index):
    viewer_colors[instance_index == object_id] = instance_color(int(object_id))

viewer_cloud = o3d.geometry.PointCloud()
viewer_cloud.points = o3d.utility.Vector3dVector(points.astype(np.float64, copy=False))
viewer_cloud.colors = o3d.utility.Vector3dVector(viewer_colors)
o3d.visualization.draw_geometries(
    [viewer_cloud], window_name=f"{scene_name} - {len(points):,} points",
    width=1400, height=900, point_show_normal=False,
)